# CJK Research Conclusion — Comparing Clip Duration Across the cjk-research Notebook Family

This notebook aggregates results from `cjk-research-2sec.ipynb`, `cjk-research-5sec.ipynb`, `cjk-research-7sec.ipynb`, and `cjk-research-11sec.ipynb` — each identical except for `CLIP_DURATION_S` (2, 5, 7, 11 seconds; all at `SAMPLE_RATE = 44100`) — and adds a fresh, matching feature-statistics sweep so every duration can be compared side by side in one place.

**What's compared:**
- Classifier accuracy / F1 (SVM, Random Forest, soft-vote ensemble) per duration — the real numbers already computed and printed in each notebook's "Classifier Sanity Checks" section, not re-run here (5-fold CV is expensive and was already done once per notebook).
- Mean value of each of the 8 hand-crafted audio features per duration — recomputed here using the exact same sampling (20 files/class, `random_state=42`) as every source notebook, since the full feature table isn't persisted outside a notebook's own kernel — only a `.head()` preview is.
- Dataset-level facts that don't vary by duration (file counts, native sample-rate distribution) — cited once from `cjk-research-2sec.ipynb`'s "Dataset At A Glance," not recomputed.

## Setup

Same imports, config, and helper functions as the `cjk-research-*sec.ipynb` family, condensed into one cell since this notebook's job is aggregation, not exploration.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import librosa
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

SAMPLE_RATE = 44100
RANDOM_SEED = 42
DURATIONS_TO_COMPARE = [2.0, 5.0, 7.0, 11.0]

REPO_ROOT = Path.cwd()
DATA_CANDIDATES = [
    REPO_ROOT / "data" / "raw" / "CatSound_originals",
    REPO_ROOT.parent / "data" / "raw" / "CatSound_originals",
    Path("../data/raw/CatSound_originals"),
]
DATA_DIR = next((c for c in DATA_CANDIDATES if c.exists()), DATA_CANDIDATES[-1])
AUDIO_EXTENSIONS = {".mp3", ".wav", ".flac", ".m4a", ".ogg"}


def is_audio_file(path: Path) -> bool:
    return path.suffix.lower() in AUDIO_EXTENSIONS


def fit_to_duration(audio: np.ndarray, sample_rate: int, duration_s: float) -> np.ndarray:
    target_length = int(round(duration_s * sample_rate))
    current_length = len(audio)
    if current_length < target_length:
        total_pad = target_length - current_length
        pad_before = total_pad // 2
        pad_after = total_pad - pad_before
        return np.pad(audio, (pad_before, pad_after), mode="constant")
    if current_length > target_length:
        total_trim = current_length - target_length
        trim_before = total_trim // 2
        return audio[trim_before:trim_before + target_length]
    return audio


def show_plotly(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

## Dataset

Rebuilds the same file inventory (`df`) as the `cjk-research-*sec.ipynb` notebooks — needed so the feature sweep below samples the exact same files, in the exact same order, per class.

In [2]:
classes = sorted([item.name for item in DATA_DIR.iterdir() if item.is_dir()])
audio_files = []
for label in classes:
    for path in sorted((DATA_DIR / label).iterdir()):
        if path.is_file() and is_audio_file(path):
            audio_files.append({"label": label, "filename": path.name, "path": path})

df = pd.DataFrame(audio_files)
print(f"{len(classes)} classes, {len(df)} audio files")

10 classes, 2953 audio files


## Feature Statistics Sweep Across Durations

For each `CLIP_DURATION_S` in {2, 5, 7, 11}, samples the same 20 files per class (`random_state=42`, matching every `cjk-research-*sec.ipynb`) and computes the mean of each of the 8 hand-crafted features. This is the one piece of analysis this notebook actually performs itself — everything else below is aggregated from already-executed results.

In [3]:
feature_columns = ["rms", "zcr", "spectral_centroid", "spectral_bandwidth", "spectral_rolloff", "mfcc_1", "mfcc_2"]

sweep_rows = []
for duration in DURATIONS_TO_COMPARE:
    feature_samples = []
    for label, frame in df.groupby("label"):
        sample = frame.sample(n=min(20, len(frame)), random_state=RANDOM_SEED)
        feature_samples.append(sample[["label", "filename", "path"]])
    feature_sample = pd.concat(feature_samples, ignore_index=True)

    rows = []
    for label, filename, path in feature_sample.itertuples(index=False, name=None):
        y, sr = librosa.load(str(path), sr=SAMPLE_RATE, mono=True)
        y = fit_to_duration(y, sr, duration)
        rows.append(
            {
                "label": label,
                "rms": float(librosa.feature.rms(y=y).mean()),
                "zcr": float(librosa.feature.zero_crossing_rate(y).mean()),
                "spectral_centroid": float(librosa.feature.spectral_centroid(y=y, sr=sr).mean()),
                "spectral_bandwidth": float(librosa.feature.spectral_bandwidth(y=y, sr=sr).mean()),
                "spectral_rolloff": float(librosa.feature.spectral_rolloff(y=y, sr=sr).mean()),
                "mfcc_1": float(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=1).mean()),
                "mfcc_2": float(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=2)[1].mean()),
            }
        )
    duration_feature_df = pd.DataFrame(rows)
    summary = duration_feature_df[feature_columns].mean()
    summary["clip_duration_s"] = duration
    sweep_rows.append(summary)
    print(f"CLIP_DURATION_S={duration}: done")

feature_sweep = pd.DataFrame(sweep_rows).set_index("clip_duration_s")
display(feature_sweep.round(4))

CLIP_DURATION_S=2.0: done


CLIP_DURATION_S=5.0: done


CLIP_DURATION_S=7.0: done


CLIP_DURATION_S=11.0: done


,rms,zcr,spectral_centroid,spectral_bandwidth,spectral_rolloff,mfcc_1,mfcc_2
clip_duration_s,,,,,,,
2.0,0.0433,0.0742,2555.8503,2621.4471,5064.3125,-403.6862,140.7765
5.0,0.0304,0.0551,1953.6469,2047.8423,3925.2469,-479.4036,104.4379
7.0,0.0228,0.0423,1502.9670,1579.2132,3025.6804,-535.5220,79.8745
11.0,0.0145,0.0270,962.4032,1011.1320,1938.1472,-602.2570,51.0932


### Feature Means by Clip Duration

In [4]:
fig = make_subplots(rows=2, cols=4, subplot_titles=feature_columns)
for i, feat in enumerate(feature_columns):
    r, c = divmod(i, 4)
    fig.add_trace(
        go.Scatter(x=feature_sweep.index, y=feature_sweep[feat], mode="lines+markers", name=feat),
        row=r + 1, col=c + 1,
    )
fig.update_layout(height=480, showlegend=False, title="Mean feature value by CLIP_DURATION_S")
fig.update_xaxes(title_text="seconds")
show_plotly(fig)

## Classifier Accuracy Across Durations

Pulled directly from the "Classifier Sanity Checks" output already executed in each `cjk-research-*sec.ipynb` — not recomputed here, since the 5-fold cross-validation was already run per notebook and is expensive to repeat.

In [5]:
classifier_results = pd.DataFrame([
    {"clip_duration_s": 2, "model": "svm_rbf", "accuracy": 0.455, "accuracy_std": 0.081, "f1_macro": 0.440},
    {"clip_duration_s": 2, "model": "random_forest", "accuracy": 0.430, "accuracy_std": 0.073, "f1_macro": 0.409},
    {"clip_duration_s": 2, "model": "soft_vote", "accuracy": 0.435, "accuracy_std": 0.073, "f1_macro": 0.413},
    {"clip_duration_s": 5, "model": "svm_rbf", "accuracy": 0.460, "accuracy_std": 0.086, "f1_macro": 0.445},
    {"clip_duration_s": 5, "model": "random_forest", "accuracy": 0.425, "accuracy_std": 0.055, "f1_macro": 0.411},
    {"clip_duration_s": 5, "model": "soft_vote", "accuracy": 0.450, "accuracy_std": 0.088, "f1_macro": 0.427},
    {"clip_duration_s": 7, "model": "svm_rbf", "accuracy": 0.440, "accuracy_std": 0.107, "f1_macro": 0.427},
    {"clip_duration_s": 7, "model": "random_forest", "accuracy": 0.420, "accuracy_std": 0.053, "f1_macro": 0.414},
    {"clip_duration_s": 7, "model": "soft_vote", "accuracy": 0.435, "accuracy_std": 0.089, "f1_macro": 0.407},
    {"clip_duration_s": 11, "model": "svm_rbf", "accuracy": 0.440, "accuracy_std": 0.093, "f1_macro": 0.428},
    {"clip_duration_s": 11, "model": "random_forest", "accuracy": 0.430, "accuracy_std": 0.037, "f1_macro": 0.416},
    {"clip_duration_s": 11, "model": "soft_vote", "accuracy": 0.415, "accuracy_std": 0.072, "f1_macro": 0.388},
])
display(classifier_results.pivot(index="clip_duration_s", columns="model", values="accuracy").round(3))

model,random_forest,soft_vote,svm_rbf
clip_duration_s,,,
2,0.430,0.435,0.455
5,0.425,0.450,0.460
7,0.420,0.435,0.440
11,0.430,0.415,0.440


### Accuracy and F1 Charts

In [6]:
acc_fig = px.line(
    classifier_results,
    x="clip_duration_s",
    y="accuracy",
    color="model",
    markers=True,
    error_y="accuracy_std",
    title="Classifier accuracy vs. clip duration (5-fold CV, error bars = fold std)",
)
acc_fig.update_layout(xaxis_title="CLIP_DURATION_S (seconds)", yaxis_title="Accuracy")
show_plotly(acc_fig)

f1_fig = px.bar(
    classifier_results,
    x="clip_duration_s",
    y="f1_macro",
    color="model",
    barmode="group",
    title="Macro F1 vs. clip duration",
)
f1_fig.update_layout(xaxis_title="CLIP_DURATION_S (seconds)", yaxis_title="F1 (macro)")
show_plotly(f1_fig)

**Reading these results:** accuracy stays in a narrow band (~0.42-0.46) across all four durations — there's no clear monotonic trend where longer or shorter clips help. Differences between durations are comparable in size to each model's own cross-validation std (0.04-0.11), meaning **duration choice alone doesn't meaningfully move classifier performance** for this hand-crafted-feature pipeline on this dataset. That matters for a future API: a shorter clip (2s) captures audio and predicts faster with no measurable accuracy cost, favoring `CLIP_DURATION_S = 2.0` over longer windows on a pure latency basis — see the earlier discussion in `cjk-research-2sec.ipynb` on why sample rate and clip duration matter for a production API.

## Dataset Context (duration-independent, cited from `cjk-research-2sec.ipynb`)

These facts don't change with `CLIP_DURATION_S` since they describe the raw dataset before any cropping/padding — see `cjk-research-2sec.ipynb` for the full analysis and interactive charts:

- **2953 files across 10 classes**, originals only (team decision 08.09.2026), no exact duplicates remaining.
- **149 segment groups / 398 files** are pieces of the same longer recording cut into pieces — must stay grouped in any train/test split, not split randomly row-by-row.
- **Native sample rate**: 98.7% of files are 44100 Hz; a 1.3% minority sits at 8000-32000 Hz. `SAMPLE_RATE = 44100` (used in every `cjk-research-*sec.ipynb` variant, including this sweep) maximizes fidelity, only upsampling that 1.3% minority rather than downsampling the dominant majority.
- **Raw duration**: mean 3.87s, p95 6.82s across the uncropped files — the basis for the historical `CLIP_DURATION_S = 7.0` recommendation, now empirically tested here against the 2/5/11s alternatives above.